# Cleaning2
## Inizializzazione ed Import

In [2]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [ ]:
file_codes = ['UCSFFSX51']     #'ADNIMERGE', 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 'UCSFFSX', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T'] #

In [ ]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


## Operazioni
- Eliminare i parametri con troppe poche righe
- Eliminare soffetti con solo 1 visita
- nuovi metadati (cofattori e fattori)

In [ ]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned2'

if os.path.isfile(new_name+'.xlsx'):
    update_new_support_file(support_file, new_name)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [ ]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    # Eliminare i parametri con troppe poche righe
    df_cleaned, file_code, new_support_file = dataCleaner.remove_param_few_subjects(df_new, file_name, prefix='cleaned/single_file')
    # --> funzione che trova i soggetti che hanno solo una visita quindi elimina quelle righe
    
    if file_code not in ['ADSP_PHC_BIOMARKER', 'PTDEMOG']:         # file solo con 1 visita, o info demog che anche una sola visita basta perchè baseline quindi da unire per ampliare il dataset ma non da usare da solo
        # Eliminare soggetti con solo 1 visita
        df_cleaned= dataCleaner.remove_sub_1visit(df_cleaned)
        # --> funzione che trova i parametri identificati da eliminare  ==> eliminare le colonne dal df

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in df_cleaned.columns]
    if to_dummy_list:
        final_df, bool_var = dataCleaner.classes_to_dummies(df_cleaned, col_list=to_dummy_list) 
    else:
        final_df = df_cleaned
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_02')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_02', updated_support_file=new_support_file)
    
    # aggiornamento metadati nel support file
    #new_support_file = dataCleaner.update_metadati_support(new_support_file) #non capisco questa funzione
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 


# Controllo file caricati

In [3]:
# verifica file caricati
search = client.query_files(
    query={'custom.level' : 'cleaned_03', 'custom.source' : 'ADNI'})
files = [x['filename'] for x in search['included_files']]
print(files)

['ADNIMERGE_25Jul2025_03.csv', 'MMSE_25Jul2025_03.csv', 'PTDEMOG_25Jul2025_03.csv', 'ADSP_PHC_BIOMARKER_25Jul2025_03.csv', 'BLCHANGE_25Jul2025_03.csv', 'DXSUM_25Jul2025_03.csv', 'UCSFFSX7_11Aug2025_03.csv', 'UCSFFSX6_11Aug2025_03.csv', 'UCSFFSX51_11_08_19_11Aug2025_03.csv', 'UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_03.csv']


In [4]:
zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

In [10]:
display(zip_files['UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_03.csv'])

,VISIT_MONTH,RID,EXAMDATE,STATUS,LVentricle,LEntorhinal,LFusiform,LHippocampus,LMidTemp,RVentricle,REntorhinal,RFusiform,RHippocampus,RMidTemp,ICV%ICV
0,0,15,2005-10-31,partial,16241,1909,7607,3510,8259,16587,2070,8392,3504,9976,100.0
1,6,15,2006-05-02,partial,16546,1906,7120,3563,7913,16693,1998,7901,3604,9874,100.0
2,11,15,2006-10-16,partial,16797,1625,7186,3585,7250,16786,1655,7915,3570,9864,100.0
3,23,15,2007-10-11,partial,17099,1867,7900,3552,7653,17558,2317,7840,3489,9575,100.0
4,42,15,2009-04-27,partial,16843,1808,7335,3488,7743,16983,1702,8078,3507,9801,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,12,1385,2009-03-12,partial,8646,1673,5974,1867,7350,8434,1482,6782,2030,7892,100.0
451,0,1387,2007-10-26,partial,39083,1188,5458,2106,6642,44362,1252,6172,2171,7430,100.0
452,5,1387,2008-03-13,partial,39771,1241,6027,2289,7406,44727,1316,6513,2515,7326,100.0
453,11,1387,2008-09-12,partial,40053,938,5705,2384,7171,45483,899,6485,2188,7486,100.0


## ADD NORMALIZATION SCALE VALUES

In [ ]:
dataCleaner = DataCleaner(support_file_path='ADNI_variables_cleaned2.xlsx')

In [ ]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'cleaned_03',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [ ]:
search

In [ ]:
metadata = client.get_metadata(
            object_name = 'cleaned/single_file/' + file_name
        )
metadata_costum = metadata['metadata']['custom']
print(metadata_costum)

In [ ]:
new_metadata = dataCleaner.get_normalization_settings(dataset)

In [ ]:
search = client.search_files(
    query={
        'custom.level' : 'cleaned_02',
        'custom.file_code': file_code
    }
)

old_metadata = search['files'][0]['custom']

In [ ]:
old_metadata['norm_scale_value'] = new_metadata

In [ ]:
# Update metadata only
result = client.update_file(
    object_name=search['files'][0]['object_name'],
    metadata=old_metadata,
)

## ADD NORMALIZATION VOLUMES VALUES

In [ ]:
dataCleaner = DataCleaner(support_file_path='ADNI_variables_cleaned2.xlsx')

In [ ]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'cleaned_02',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [ ]:
new_metadata = dataCleaner.get_normalization_settings(dataset, file_name='volume_values_settings.json')

In [ ]:
search = client.search_files(
    query={
        'custom.level' : 'cleaned_02',
        'custom.file_code': file_code
    }
)

old_metadata = search['files'][0]['custom']

In [ ]:
old_metadata['volume_norm_values'] = new_metadata

In [ ]:
# Update metadata only
result = client.update_file(
    object_name=search['files'][0]['object_name'],
    metadata=old_metadata,
)